This will walk you through setting up your own API and using it to retrieve data from a database

## Request Types

In [2]:
# In an API, you can define different types of HTTP requests for your endpoints. The most common request types are:
# - GET: Retrieve data from the server.
# - POST: Send data to the server to create a new resource.
# - PUT: Update an existing resource on the server.
# - DELETE: Remove a resource from the server.

# For example, with the UHC API, a GET request would be used to get claim status, but a POST request would be used to submit a new claim.

# With APIs, they follow the "CRUD" operations: Create, Read, Update, Delete.
# - Create: Corresponds to POST requests.
# - Read: Corresponds to GET requests.
# - Update: Corresponds to PUT requests.
# - Delete: Corresponds to DELETE requests.

# Setting up an API application

In [3]:
from fastapi import FastAPI

app = FastAPI()

# basic get request
@app.get("/")
def read_root():
    return {"Hello": "World"}

# the above created an endpoint that responds to GET requests at the root URL ("/")

# Running your API
Running the FastAPI app requires more than just running the python file. You need to establish a server to run the app, which can be done by using a server like Uvicorn.  To run this server, you can use the command: 'uv run fastapi dev' from your terminal

Since fastapi and jupyter do not play well together, so the application code referenced in this notebook can be found in the Tutorials/api/main.py file. to run this, run:
uv run fastapi dev Tutorials/api/main.py 

if you do not set a file path, Uvicorn will look for an app instance in the current directory's "main.py" file.

Once run, you should see "Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)" in your terminal.

To test if its working, open your web browser and navigate to http://127.0.0.1:8000. You should see a JSON response: {"Hello": "World"}

** Your web browser treats visiting a URL as a GET request. **

If you close out of VSCode or press CTRL+C while in the terminal, the server will shut down. This server is ONLY accesible to you and your computer.

# More Endpoints

In [4]:
# The uvicorn server is constantly looking for updates to the file and will automatically reload the application when changes are detected.

# Lets add a new endpoint to our FastAPI application that will be used to return a greeting message to the user.

# Paste the below code into the Tutorials/api/main.py file.

@app.get("/greet/{name}")
def greet_user(name):
    return {"message": f"Hello, {name}!"}

# You will see in the terminal that changes were detected and the application was automatically reloaded.

# The URL endpoint takes the string "name" as a parameter and returns a personalized greeting message. To test, navigate to http://127.0.0.1:8000/greet/David in your web browser.

# FastAPI takes the URL parameter and passes it to the function as an argument. As such, the field "name" in the URL is mapped to the parameter "name" in the function and must be the same.

# if we wanted to have multiple parameters, we could define them in the URL and the function signature accordingly. For example, paste this code into the Tutorials/api/main.py file and navigate to http://127.0.0.1:8000/greet/David/21 in your web browser.

# additionally, by defining the data type of the parameter, FastAPI will automatically validate the input
@app.get("/greet/{name}/{age}")
def greet_user_with_age(name: str, age: int):
    return {"message": f"Hello, {name}! You are {age} years old."}

# Documentation

In [5]:
# FastAPI also automatically generates API documentation for you
# You can access it by navigating to http://127.0.0.1:8000/docs in your web browser.

# This allows you to test the API endpoints directly from the browser and preview the results. 

# You can also add documentation within the function docstrings, which will be displayed in the automatically generated API documentation. Replace the greet_user_with_age function in main.py and refresh the docs to see:
from fastapi import status, HTTPException

@app.get("/greet/{name}/{age}")
def greet_user_with_age(name: str, age: int):
    """
    Greet a user with their name and age.

    - **name**: The name of the user
    - **age**: The age of the user
    """
    return {"message": f"Hello, {name}! You are {age} years old."}

# Status Responses

In [6]:
# API's follow RESTful guidelines and should always follow standard conventions for URL structure, HTTP methods, and response codes.

# Standard Response Codes:
# 200: OK - The request was successful
# 201: Created - The request was successful and a resource was created
# 400: Bad Request - The request was invalid or cannot be served
# 401: Unauthorized - The request requires authentication
# 403: Forbidden - The server understood the request, but refuses to authorize it
# 404: Not Found - The requested resource could not be found
# 500: Internal Server Error - The server encountered an error and could not complete the request


# When you design an API, it's important to use these standard response codes to indicate the outcome of the request. To do this, you can use the `status_code` parameter in your FastAPI route decorators to define what returns by default. Overwrite the greet_user_with_age function in main.py:
@app.get("/greet/{name}/{age}", status_code=status.HTTP_200_OK)
def greet_user_with_age(name: str, age: int):
    """
    Greet a user with their name and age.

    - **name**: The name of the user
    - **age**: The age of the user
    """
    return {"message": f"Hello, {name}! You are {age} years old."}


# lets say you want to return a different status code if the user is under 18
@app.get("/greet/{name}/{age}", status_code=status.HTTP_200_OK)
def greet_user_with_age(name: str, age: int):
    # a 200 response will be returned by default if there are no errors with the API. to throw an API level exception, use HTTPException and raise it
    """Greet a user with their name and age. If the user is under 18, a 403 Forbidden response will be returned.

    - **name**: The name of the user
    - **age**: The age of the user
    """
    if age < 18:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Forbidden: You must be at least 18 years old"
        )
    return {"message": f"Hello, {name}! You are {age} years old."}

# overwrite the greet_user_with_age function in main.py with this new version to handle age restrictions and test it in the API documentation.

# Query Parameters

In [7]:
# in the above, we used path parameters (defined in the URL) to capture user input. Another option is to use query parameters, which are appended to the URL after a question mark (?).

# For example, the URL /greet?name=David would pass the value "David" to the name parameter in the function.

# add the following into the main.py file:

@app.get("/ages")
def get_age(min_age: int):
    fake_items_db = [{"name": "David", "age": 25}, {"name": "Bar", "age": 30}, {"name": "Baz", "age": 20}, {"name": "Jake", "age": 17}]
    return [item for item in fake_items_db if item["age"] >= min_age]

# to test this endpoint, navigate to the following URL in your web browser: http://127.0.0.1:8000/ages?min_age=21

# you can also set a default value for the query parameter by assigning a value in the function signature, e.g., min_age: int = 18
@app.get("/ages")
def get_age(min_age: int = 18):
    fake_items_db = [{"name": "David", "age": 25}, {"name": "Bar", "age": 30}, {"name": "Baz", "age": 20}, {"name": "Jake", "age": 17}]
    return [item for item in fake_items_db if item["age"] >= min_age]

# Connecting the database

In [8]:
# lets create some endpoints that will pull data from your database. delete everything in main.py and paste the following:

from fastapi import FastAPI, HTTPException, status
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from api.models import LegoSet

app = FastAPI()

connection_string = "sqlite:///lego_sets.db"
engine = create_engine(connection_string)

@app.get("/by-set-id/{set_id}")
def get_set_by_id(set_id: str):
    """Get a LEGO set by its ID.

   set_id (str): The ID of the LEGO set.
    """
    with Session(engine) as session:
        result = session.query(LegoSet).filter(LegoSet.set_id == set_id).first()
        if not result:
            raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="LEGO set not found")
        return result.__dict__
    
# use docs viewer to test the API. id = 1 should return an error, id = 345-3 should return a valid LEGO set

# Lets create some CRUD

## CREATE

In [ ]:
# Since your browser can only send GET requests, we cannot send information to the server. In order to send information, we need to send a "request body"

# in this case, the request body we want to send is a LegoSet object. however, users will not have insight into the internal structure of the LegoSet model, so we need to use Pydantic models to define the request body.

# instead of now defining column values, we are defining what attributes the request body should have.

from pydantic import BaseModel
from typing import Optional

class LegoSetModel(BaseModel):
    set_id: str
    name: str
    year: int
    theme: str
    subtheme: Optional[str] = None # similar to the column definition, we need to tell python which fields are optional
    themeGroup: Optional[str] = None
    category: Optional[str] = None
    pieces: Optional[int] = None
    minifigs: Optional[int] = None
    agerange_min: Optional[int] = None
    US_retailPrice: Optional[float] = None
    bricksetURL: Optional[str] = None
    thumbnailURL: Optional[str] = None
    imageURL: Optional[str] = None
    
@app.post("/create-lego-set", status_code=status.HTTP_201_CREATED)
def create_set(lego_set: LegoSetModel):
    """Create a new LEGO set.

    lego_set (LegoSetModel): The LEGO set to create.
    """
    with Session(engine) as session:
        lego_set_dict = lego_set.model_dump()
        # model_dump will convert the object into a dictionary
        lego_set_obj = LegoSet(**lego_set_dict) # convert dictionary to LegoSet object
        session.add(lego_set_obj)
        session.commit()
        return lego_set_dict
    
# if you add the above and refresh the docmentation, youll see "LegoSetModel" underneath the "schemas" section

# instead of defining a pydantic model, we could also take in a dictionary directly or keyword arguments, but you would then need to add your own validation logic to ensure the data is correct and is not recommended

# you can see how redundant this is in the below example:
@app.post("/create-lego-set-long")
def create_set_long(set_id: str, name: str, year: int, theme: str, subtheme: Optional[str] = None, themeGroup: Optional[str] = None, category: Optional[str] = None, pieces: Optional[int] = None, minifigs: Optional[int] = None, agerange_min: Optional[int] = None, US_retailPrice: Optional[float] = None, bricksetURL: Optional[str] = None, thumbnailURL: Optional[str] = None, imageURL: Optional[str] = None):
    """Create a new LEGO set.

    set_id (str): The ID of the LEGO set.
    name (str): The name of the LEGO set.
    year (int): The year the LEGO set was released.
    theme (str): The theme of the LEGO set.
    subtheme (Optional[str]): The subtheme of the LEGO set.
    themeGroup (Optional[str]): The theme group of the LEGO set.
    category (Optional[str]): The category of the LEGO set.
    pieces (Optional[int]): The number of pieces in the LEGO set.
    minifigs (Optional[int]): The number of minifigures in the LEGO set.
    agerange_min (Optional[int]): The minimum age range for the LEGO set.
    US_retailPrice (Optional[float]): The US retail price of the LEGO set.
    bricksetURL (Optional[str]): The URL of the LEGO set on Brickset.
    thumbnailURL (Optional[str]): The URL of the thumbnail image of the LEGO set.
    imageURL (Optional[str]): The URL of the image of the LEGO set.
    """
    lego_set_dict = {
        "set_id": set_id,
        "name": name,
        "year": year,
        "theme": theme,
        "subtheme": subtheme,
        "themeGroup": themeGroup,
        "category": category,
        "pieces": pieces,
        "minifigs": minifigs,
        "agerange_min": agerange_min,
        "US_retailPrice": US_retailPrice,
        "bricksetURL": bricksetURL,
        "thumbnailURL": thumbnailURL,
        "imageURL": imageURL
    }
    
    required_fields = ["set_id", "name", "year", "theme"]
    if any(lego_set_dict[field] is None for field in required_fields):
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Missing required fields: {', '.join(field for field in required_fields if lego_set_dict[field] is None)}"
        )
        
    with Session(engine) as session:
        session.add(lego_set_dict)
        session.commit()
        return lego_set_dict

## READ

In [ ]:
@app.get("/get-lego-set/{set_id}", status_code=status.HTTP_200_OK)
def get_lego_set(set_id: str):
    """Get a LEGO set by its ID.

    set_id (str): The ID of the LEGO set.
    """
    with Session(engine) as session:
        lego_set = session.get(LegoSet, set_id)
        if not lego_set:
            raise HTTPException(
                status_code=status.HTTP_404_NOT_FOUND,
                detail=f"LEGO set with ID {set_id} not found"
            )
        return lego_set

## UPDATE

In [ ]:
# update a record
@app.put("/update-lego-set/{set_id}", status_code=status.HTTP_200_OK)
def update_lego_set(set_id: str, lego_set: LegoSetModel):
    """Update a LEGO set by its ID.

    set_id (str): The ID of the LEGO set.
    lego_set (LegoSetModel): The updated LEGO set data.
    """
    with Session(engine) as session:
        existing_lego_set = session.get(LegoSet, set_id)
        if not existing_lego_set:
            raise HTTPException(
                status_code=status.HTTP_404_NOT_FOUND,
                detail=f"LEGO set with ID {set_id} not found"
            )
        # iterate through the fields of the incoming LEGO set and update the existing LEGO set
        for key, value in lego_set.model_dump().items():
            setattr(existing_lego_set, key, value)
        session.commit()
        return existing_lego_set

## DELETE

In [ ]:
# delete a row
@app.delete("/delete-lego-set/{set_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_lego_set(set_id: str):
    """Delete a LEGO set by its ID.

    set_id (str): The ID of the LEGO set.
    """
    with Session(engine) as session:
        lego_set = session.get(LegoSet, set_id)
        if not lego_set:
            raise HTTPException(
                status_code=status.HTTP_404_NOT_FOUND,
                detail=f"LEGO set with ID {set_id} not found"
            )
        session.delete(lego_set)
        session.commit()
        return {"detail": f"LEGO set with ID {set_id} deleted"}

# Try it Out

Create CRUD API endpoints for your video_games_data database in main.py. Add your pydantic model into models.py

For a harder challenge, create endpoints for common ways to filter (ie filter by year, by console, etc)